In [3]:
import os, shutil, numpy as np, tensorflow as tf, mlflow
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
!pip install mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 95.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.5 MB/s eta 0:00:00


In [9]:
all_normal, all_pneumonia = [], []
for split in ["train", "val", "test"]:
    for f in os.listdir(f"/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/{split}/NORMAL"):
        all_normal.append(f"/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/{split}/NORMAL/{f}")
    for f in os.listdir(f"/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/{split}/PNEUMONIA"):
        all_pneumonia.append(f"/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/{split}/PNEUMONIA/{f}")

print(f"Total NORMAL:    {len(all_normal)}")
print(f"Total PNEUMONIA: {len(all_pneumonia)}")

n_train, n_temp = train_test_split(all_normal,    test_size=0.2, random_state=42)
n_val,   n_test = train_test_split(n_temp,        test_size=0.5, random_state=42)
p_train, p_temp = train_test_split(all_pneumonia, test_size=0.2, random_state=42)
p_val,   p_test = train_test_split(p_temp,        test_size=0.5, random_state=42)

print(f"\nNew split:")
print(f"Train → NORMAL: {len(n_train)}  PNEUMONIA: {len(p_train)}")
print(f"Val   → NORMAL: {len(n_val)}    PNEUMONIA: {len(p_val)}")
print(f"Test  → NORMAL: {len(n_test)}   PNEUMONIA: {len(p_test)}")

Total NORMAL:    1583
Total PNEUMONIA: 4273

New split:
Train → NORMAL: 1266  PNEUMONIA: 3418
Val   → NORMAL: 158    PNEUMONIA: 427
Test  → NORMAL: 159   PNEUMONIA: 428


In [10]:
BASE = "/kaggle/working/data_v3"
SRC_BASE = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

def copy_files(file_list, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for src in file_list:
        shutil.copy2(src, os.path.join(dst_dir, os.path.basename(src)))

copy_files(n_train, f"{BASE}/train/NORMAL")
copy_files(p_train, f"{BASE}/train/PNEUMONIA")
copy_files(n_val,   f"{BASE}/val/NORMAL")
copy_files(p_val,   f"{BASE}/val/PNEUMONIA")
copy_files(n_test,  f"{BASE}/test/NORMAL")
copy_files(p_test,  f"{BASE}/test/PNEUMONIA")

print("Files copied:")
for split in ["train","val","test"]:
    for cls in ["NORMAL","PNEUMONIA"]:
        print(f"  {split}/{cls}: {len(os.listdir(f'{BASE}/{split}/{cls}'))}")

Files copied:
  train/NORMAL: 1266
  train/PNEUMONIA: 3418
  val/NORMAL: 158
  val/PNEUMONIA: 427
  test/NORMAL: 159
  test/PNEUMONIA: 428


In [11]:
IMG_SIZE, BATCH_SIZE = (224, 224), 32
MODEL_DIR = "/kaggle/working/models_v3"
os.makedirs(MODEL_DIR, exist_ok=True)

train_gen    = ImageDataGenerator(
    rescale=1./255, rotation_range=15, zoom_range=0.1,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, fill_mode="nearest"
)
val_test_gen = ImageDataGenerator(rescale=1./255)

train = train_gen.flow_from_directory(f"{BASE}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="binary")
val   = val_test_gen.flow_from_directory(f"{BASE}/val",   target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="binary")
test  = val_test_gen.flow_from_directory(f"{BASE}/test",  target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="binary")
print("Class indices:", train.class_indices)

Found 4684 images belonging to 2 classes.
Found 585 images belonging to 2 classes.
Found 587 images belonging to 2 classes.
Class indices: {'NORMAL': 0, 'PNEUMONIA': 1}


In [12]:
base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights="imagenet")
base.trainable = False
x   = GlobalAveragePooling2D()(base.output)
x   = BatchNormalization()(x)
x   = Dense(256, activation="relu")(x)
x   = Dropout(0.4)(x)
x   = Dense(64,  activation="relu")(x)
x   = Dropout(0.2)(x)
out = Dense(1,   activation="sigmoid")(x)
model = Model(base.input, out)
model.compile(
    optimizer=Adam(1e-4), loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=train.classes)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

I0000 00:00:1776485424.247459      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Class weights: {0: np.float64(1.8499210110584519), 1: np.float64(0.6851960210649503)}


In [13]:
mlflow.set_tracking_uri("/kaggle/working/mlruns")
mlflow.set_experiment("chest-xray-v3-proper-split")

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_auc"),
    ReduceLROnPlateau(factor=0.5, patience=3, monitor="val_loss"),
    ModelCheckpoint(f"{MODEL_DIR}/best_model_v3.h5", save_best_only=True, monitor="val_auc"),
]

with mlflow.start_run(run_name="v3-phase1-frozen"):
    print("=== Phase 1: Frozen base ===")
    h1 = model.fit(train, validation_data=val, epochs=15,
                   callbacks=callbacks, class_weight=class_weight_dict)
    mlflow.log_metric("best_val_auc", max(h1.history["val_auc"]))
print("Phase 1 done!")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/18 04:10:36 INFO mlflow.tracking.fluent: Experiment with name 'chest-xray-v3-proper-split' does not exist. Creating a new experiment.


=== Phase 1: Frozen base ===


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1776485444.098342     185 service.cc:152] XLA service 0x78aa90001da0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776485444.098385     185 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776485445.343885     185 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776485452.548276     185 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - accuracy: 0.6645 - auc: 0.7865 - loss: 0.5711 - precision: 0.8919 - recall: 0.6025

147/147 ━━━━━━━━━━━━━━━━━━━━ 117s 691ms/step - accuracy: 0.6653 - auc: 0.7872 - loss: 0.5701 - precision: 0.8922 - recall: 0.6035 - val_accuracy: 0.9282 - val_auc: 0.9836 - val_loss: 0.2418 - val_precision: 0.9873 - val_recall: 0.9133 - learning_rate: 1.0000e-04
Epoch 2/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 83s 568ms/step - accuracy: 0.8769 - auc: 0.9519 - loss: 0.2792 - precision: 0.9661 - recall: 0.8613 - val_accuracy: 0.9009 - val_auc: 0.9887 - val_loss: 0.2418 - val_precision: 0.9946 - val_recall: 0.8689 - learning_rate: 1.0000e-04
Epoch 3/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 81s 553ms/step - accuracy: 0.8952 - auc: 0.9645 - loss: 0.2377 - precision: 0.9685 - recall: 0.8866 - val_accuracy: 0.9043 - val_auc: 0.9913 - val_loss: 0.2214 - val_precision: 0.9973 - val_recall: 0.8712 - learning_rate: 1.0000e-04
Epoch 4/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 81s 552ms/step - accuracy: 0.9083 - auc: 0.9750 - loss: 0.2032 - precision: 0.9741 - recall: 0.8994 - val_accuracy: 0.8991 - val_auc: 0.9916 - val_loss

In [14]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5), loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

with mlflow.start_run(run_name="v3-phase2-finetune"):
    print("=== Phase 2: Fine-tuning ===")
    h2 = model.fit(train, validation_data=val, epochs=10,
                   callbacks=callbacks, class_weight=class_weight_dict)

    results = model.evaluate(test)
    names   = ["loss","accuracy","auc","precision","recall"]
    metrics = dict(zip(names, results))
    for k,v in metrics.items():
        print(f"  {k}: {v:.4f}")

    mlflow.log_metrics({f"test_{k}": v for k,v in metrics.items()})
    model.save(f"{MODEL_DIR}/mobilenetv2_v3_final.h5")
    print("Model saved!")

=== Phase 2: Fine-tuning ===
Epoch 1/10


2026-04-18 04:33:02.116849: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 04:33:02.314021: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 71/147 ━━━━━━━━━━━━━━━━━━━━ 39s 515ms/step - accuracy: 0.8732 - auc: 0.9253 - loss: 0.4290 - precision: 0.8992 - recall: 0.9320

2026-04-18 04:33:47.308542: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 04:33:47.504215: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


147/147 ━━━━━━━━━━━━━━━━━━━━ 117s 669ms/step - accuracy: 0.8860 - auc: 0.9386 - loss: 0.3717 - precision: 0.9166 - recall: 0.9297 - val_accuracy: 0.9402 - val_auc: 0.9918 - val_loss: 0.1453 - val_precision: 0.9356 - val_recall: 0.9859 - learning_rate: 1.0000e-05
Epoch 2/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 81s 554ms/step - accuracy: 0.9141 - auc: 0.9726 - loss: 0.2173 - precision: 0.9727 - recall: 0.9063 - val_accuracy: 0.9419 - val_auc: 0.9887 - val_loss: 0.1603 - val_precision: 0.9338 - val_recall: 0.9906 - learning_rate: 1.0000e-05
Epoch 3/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 81s 551ms/step - accuracy: 0.9302 - auc: 0.9792 - loss: 0.1866 - precision: 0.9766 - recall: 0.9264 - val_accuracy: 0.9487 - val_auc: 0.9883 - val_loss: 0.1471 - val_precision: 0.9421 - val_recall: 0.9906 - learning_rate: 1.0000e-05
Epoch 4/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 80s 546ms/step - accuracy: 0.9285 - auc: 0.9808 - loss: 0.1716 - precision: 0.9771 - recall: 0.9230 - val_accuracy: 0.9538 - val_auc: 0.9912 - val_loss

  loss: 0.1982
  accuracy: 0.9302
  auc: 0.9762
  precision: 0.9216
  recall: 0.9883
Model saved!


In [15]:
test_check  = val_test_gen.flow_from_directory(
    f"{BASE}/test", target_size=(224,224),
    batch_size=1, class_mode="binary", shuffle=False
)
preds       = model.predict(test_check, verbose=1)
pred_labels = (preds >= 0.5).astype(int).flatten()
true_labels = test_check.classes

normal_acc    = np.mean(pred_labels[true_labels == 0] == 0)
pneumonia_acc = np.mean(pred_labels[true_labels == 1] == 1)

print(f"\nResults:")
print(f"NORMAL    accuracy: {normal_acc*100:.1f}%")
print(f"PNEUMONIA accuracy: {pneumonia_acc*100:.1f}%")
print(f"Target: NORMAL >75%  PNEUMONIA >95%")

Found 587 images belonging to 2 classes.
587/587 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step

Results:
NORMAL    accuracy: 77.4%
PNEUMONIA accuracy: 98.8%
Target: NORMAL >75%  PNEUMONIA >95%


In [16]:
shutil.make_archive("/kaggle/working/models_v3_backup", "zip", MODEL_DIR)
print("✅ Done! Download models_v3_backup.zip from the Output panel on the right →")

✅ Done! Download models_v3_backup.zip from the Output panel on the right →
